In [1]:
import duckdb
import pandas as pd
conn = duckdb.connect('../datasets/db/adventureworks.duckdb')

In [2]:
conn.execute("SHOW TABLES").fetchdf()

,name
0,calendar_lookup
1,customer_lookup
2,product_categories_lookup
3,product_lookup
4,product_subcategories_lookup
5,returns_data
6,sales_2020
7,sales_2021
8,sales_2022
9,territory_lookup


# 1. Data Cleaning

## 1. Tables Overview

In [28]:
conn.execute("SHOW TABLES").fetchdf()

,name
0,calendar_lookup
1,customer_lookup
2,product_categories_lookup
3,product_lookup
4,product_subcategories_lookup
5,returns_data
6,sales_2020
7,sales_2021
8,sales_2022
9,territory_lookup


In [29]:
qry = """ 
    SELECT 'calendar_lookup' AS table_name, COUNT(*) AS row_count FROM calendar_lookup
    UNION ALL SELECT 'customer_lookup' AS table_name, COUNT(*) AS row_count FROM customer_lookup
    UNION ALL SELECT 'product_categories_lookup' AS table_name, COUNT(*) AS row_count FROM product_categories_lookup
    UNION ALL SELECT 'product_lookup' AS table_name, COUNT(*) AS row_count FROM product_lookup
    UNION ALL SELECT 'product_subcategories_lookup' AS table_name, COUNT(*) AS row_count FROM product_subcategories_lookup
    UNION ALL SELECT 'returns_data' AS table_name, COUNT(*) AS row_count FROM returns_data
    UNION ALL SELECT 'sales_2021' AS table_name, COUNT(*) AS row_count FROM sales_2021
    UNION ALL SELECT 'sales_2022' AS table_name, COUNT(*) AS row_count FROM sales_2022
    UNION ALL SELECT 'territory_lookup' AS table_name, COUNT(*) AS row_count FROM territory_lookup
ORDER BY table_name;
"""
conn.execute(qry).fetch_df()


,table_name,row_count
0,calendar_lookup,912
1,customer_lookup,18154
2,product_categories_lookup,4
3,product_lookup,293
4,product_subcategories_lookup,37
5,returns_data,1809
6,sales_2021,23935
7,sales_2022,29481
8,territory_lookup,10


## 2. Missing Data

### 1. Calendar Lookup Table

In [42]:
qry = """ 
    DESCRIBE calendar_lookup;
"""

conn.execute(qry).fetch_df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None


In [43]:
qry = """ 
    SELECT * FROM calendar_lookup LIMIT 5;
"""

conn.execute(qry).fetch_df()

,date
0,2020-01-01
1,2020-01-02
2,2020-01-03
3,2020-01-04
4,2020-01-05


In [46]:
qry = """ 
    SELECT COUNT(*) AS total_rows,
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS missing_date
    FROM calendar_lookup;
"""

conn.execute(qry).fetch_df()

,total_rows,missing_date
0,912,0.0


### 2. Customer Lookup Table

In [47]:
qry = """ 
    DESCRIBE customer_lookup;
"""

conn.execute(qry).fetch_df()

,column_name,column_type,null,key,default,extra
0,customerkey,VARCHAR,YES,None,None,None
1,prefix,VARCHAR,YES,None,None,None
2,firstname,VARCHAR,YES,None,None,None
3,lastname,VARCHAR,YES,None,None,None
4,birthdate,DATE,YES,None,None,None
5,maritalstatus,VARCHAR,YES,None,None,None
6,gender,VARCHAR,YES,None,None,None
7,emailaddress,VARCHAR,YES,None,None,None
8,annualincome,BIGINT,YES,None,None,None
9,totalchildren,BIGINT,YES,None,None,None


In [48]:
qry = """ 
    SELECT COUNT(*) AS total_rows,
        SUM(CASE WHEN customerkey IS NULL THEN 1 ELSE 0 END) AS customerkey_missing,
        SUM(CASE WHEN prefix IS NULL THEN 1 ELSE 0 END) AS prefix_missing,
        SUM(CASE WHEN firstname IS NULL THEN 1 ELSE 0 END) AS firstname_missing,
        SUM(CASE WHEN lastname IS NULL THEN 1 ELSE 0 END) AS lastname_missing,
        SUM(CASE WHEN birthdate IS NULL THEN 1 ELSE 0 END) AS birthdate_missing,
        SUM(CASE WHEN maritalstatus IS NULL THEN 1 ELSE 0 END) AS maritalstatus_missing,
        SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END) AS gender_missing,
        SUM(CASE WHEN emailaddress IS NULL THEN 1 ELSE 0 END) AS emailaddress_missing,
        SUM(CASE WHEN annualincome IS NULL THEN 1 ELSE 0 END) AS annualincome_missing,
        SUM(CASE WHEN totalchildren IS NULL THEN 1 ELSE 0 END) AS totalchildren_missing,
        SUM(CASE WHEN educationlevel IS NULL THEN 1 ELSE 0 END) AS educationlevel_missing,
        SUM(CASE WHEN occupation IS NULL THEN 1 ELSE 0 END) AS occupation_missing,
        SUM(CASE WHEN homeowner IS NULL THEN 1 ELSE 0 END) AS homeowner_missing
    FROM customer_lookup;
"""

conn.execute(qry).fetch_df()

,total_rows,customerkey_missing,prefix_missing,firstname_missing,lastname_missing,birthdate_missing,maritalstatus_missing,gender_missing,emailaddress_missing,annualincome_missing,totalchildren_missing,educationlevel_missing,occupation_missing,homeowner_missing
0,18064,1.0,131.0,6.0,6.0,3.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0


In [54]:
qry = """ 
    SELECT prefix FROM customer_lookup LIMIT 5;
"""

conn.execute(qry).fetch_df()

,prefix
0,MR.
1,MR.
2,MR.
3,MS.
4,MRS.


In [56]:
qry = """ 
    SELECT occupation FROM customer_lookup LIMIT 5;
"""

conn.execute(qry).fetch_df()

,occupation
0,Professional
1,Professional
2,Professional
3,Professional
4,Professional


In [ ]:
# 1. Delete missing customerkey
qry = """ 
    DELETE FROM customer_lookup WHERE customerkey IS NULL;
"""

conn.execute(qry).fetch_df()

,Count
0,1


In [59]:
# 2. Update annualincome NULL with AVG
qry = """ 
    UPDATE customer_lookup
        SET annualincome = (
            SELECT AVG(annualincome)
            FROM customer_lookup
            WHERE annualincome IS NOT NULL
        )
    WHERE annualincome IS NULL;
"""

conn.execute(qry).fetch_df()

,Count
0,5


In [60]:
# 3. Update occupation NULL with mode (the most frequent value)
qry = """ 
    UPDATE customer_lookup
        SET occupation = (
            SELECT occupation
            FROM customer_lookup
            WHERE occupation IS NOT NULL
            GROUP BY occupation
            ORDER BY COUNT(*) DESC
            LIMIT 1
        )
    WHERE occupation IS NULL;
"""

conn.execute(qry).fetch_df()


,Count
0,5


In [61]:
# Verrification
qry = """ 
    SELECT COUNT(*) AS total_rows,
        SUM(CASE WHEN customerkey IS NULL THEN 1 ELSE 0 END) AS customerkey_missing,
        SUM(CASE WHEN prefix IS NULL THEN 1 ELSE 0 END) AS prefix_missing,
        SUM(CASE WHEN firstname IS NULL THEN 1 ELSE 0 END) AS firstname_missing,
        SUM(CASE WHEN lastname IS NULL THEN 1 ELSE 0 END) AS lastname_missing,
        SUM(CASE WHEN birthdate IS NULL THEN 1 ELSE 0 END) AS birthdate_missing,
        SUM(CASE WHEN maritalstatus IS NULL THEN 1 ELSE 0 END) AS maritalstatus_missing,
        SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END) AS gender_missing,
        SUM(CASE WHEN emailaddress IS NULL THEN 1 ELSE 0 END) AS emailaddress_missing,
        SUM(CASE WHEN annualincome IS NULL THEN 1 ELSE 0 END) AS annualincome_missing,
        SUM(CASE WHEN totalchildren IS NULL THEN 1 ELSE 0 END) AS totalchildren_missing,
        SUM(CASE WHEN educationlevel IS NULL THEN 1 ELSE 0 END) AS educationlevel_missing,
        SUM(CASE WHEN occupation IS NULL THEN 1 ELSE 0 END) AS occupation_missing,
        SUM(CASE WHEN homeowner IS NULL THEN 1 ELSE 0 END) AS homeowner_missing
    FROM customer_lookup;
"""

conn.execute(qry).fetch_df()

,total_rows,customerkey_missing,prefix_missing,firstname_missing,lastname_missing,birthdate_missing,maritalstatus_missing,gender_missing,emailaddress_missing,annualincome_missing,totalchildren_missing,educationlevel_missing,occupation_missing,homeowner_missing
0,18063,0.0,130.0,5.0,5.0,2.0,5.0,5.0,5.0,0.0,5.0,5.0,0.0,5.0


<p>Other missing data are left alone except for customerkey, annualincome, and occupation because there is a <strong>possibility</strong> they will be used in the analysis phase</p>


### 3. Product Categories Lookup Table

In [3]:
qry = """ 
    DESCRIBE product_categories_lookup;
"""

conn.execute(qry).fetch_df()

,column_name,column_type,null,key,default,extra
0,productcategorykey,BIGINT,YES,None,None,None
1,categoryname,VARCHAR,YES,None,None,None


In [5]:
qry = """ 
    SELECT COUNT(*) AS total_rows,
    SUM(CASE WHEN productcategorykey IS NULL THEN 1 ELSE 0 END) AS productcategorykey_missing,
    SUM(CASE WHEN categoryname IS NULL THEN 1 ELSE 0 END) AS categoryname_missing
    FROM product_categories_lookup;
"""

conn.execute(qry).fetch_df()

,total_rows,productcategorykey_missing,categoryname_missing
0,4,0.0,0.0


### 4. Product Lookup Table

In [6]:
qry = """ 
    DESCRIBE product_lookup;
"""

conn.execute(qry).fetch_df()

,column_name,column_type,null,key,default,extra
0,productkey,BIGINT,YES,None,None,None
1,productsubcategorykey,BIGINT,YES,None,None,None
2,productsku,VARCHAR,YES,None,None,None
3,productname,VARCHAR,YES,None,None,None
4,modelname,VARCHAR,YES,None,None,None
5,productdescription,VARCHAR,YES,None,None,None
6,productcolor,VARCHAR,YES,None,None,None
7,productsize,VARCHAR,YES,None,None,None
8,productstyle,VARCHAR,YES,None,None,None
9,productcost,DOUBLE,YES,None,None,None


In [7]:
qry = """ 
    SELECT COUNT(*) AS total_rows,
    SUM(CASE WHEN productkey IS NULL THEN 1 ELSE 0 END) AS productkey_missing,
    SUM(CASE WHEN productsubcategorykey IS NULL THEN 1 ELSE 0 END) AS productsubcategorykey_missing,
    SUM(CASE WHEN productsku IS NULL THEN 1 ELSE 0 END) AS productsku_missing,
    SUM(CASE WHEN productname IS NULL THEN 1 ELSE 0 END) AS productname_missing,
    SUM(CASE WHEN modelname IS NULL THEN 1 ELSE 0 END) AS modelname_missing,
    SUM(CASE WHEN productdescription IS NULL THEN 1 ELSE 0 END) AS productdescription_missing,
    SUM(CASE WHEN productcolor IS NULL THEN 1 ELSE 0 END) AS productcolor_missing,
    SUM(CASE WHEN productsize IS NULL THEN 1 ELSE 0 END) AS productsize_missing,
    SUM(CASE WHEN productstyle IS NULL THEN 1 ELSE 0 END) AS productstyle_missing,
    SUM(CASE WHEN productcost IS NULL THEN 1 ELSE 0 END) AS productcost_missing,
    SUM(CASE WHEN productprice IS NULL THEN 1 ELSE 0 END) AS productprice_missing
    FROM product_lookup;
"""

conn.execute(qry).fetch_df()

,total_rows,productkey_missing,productsubcategorykey_missing,productsku_missing,productname_missing,modelname_missing,productdescription_missing,productcolor_missing,productsize_missing,productstyle_missing,productcost_missing,productprice_missing
0,293,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 5. Product SubCategories Lookup Table

In [8]:
conn.execute("DESCRIBE product_subcategories_lookup").fetch_df()

,column_name,column_type,null,key,default,extra
0,productsubcategorykey,BIGINT,YES,None,None,None
1,subcategoryname,VARCHAR,YES,None,None,None
2,productcategorykey,BIGINT,YES,None,None,None


In [9]:
qry = """ 
    SELECT COUNT(*) AS total_rows,
    SUM(CASE WHEN productsubcategorykey IS NULL THEN 1 ELSE 0 END) AS productsubcategorykey_missing,
    SUM(CASE WHEN subcategoryname IS NULL THEN 1 ELSE 0 END) AS subcategoryname_missing,
    SUM(CASE WHEN productcategorykey IS NULL THEN 1 ELSE 0 END) AS productcategorykey_missing
    FROM product_subcategories_lookup;
"""

conn.execute(qry).fetch_df()

,total_rows,productsubcategorykey_missing,subcategoryname_missing,productcategorykey_missing
0,37,0.0,0.0,0.0


### 6. Returns Data Table

In [10]:
conn.execute("DESCRIBE returns_data").fetch_df()

,column_name,column_type,null,key,default,extra
0,returndate,DATE,YES,None,None,None
1,territorykey,BIGINT,YES,None,None,None
2,productkey,BIGINT,YES,None,None,None
3,returnquantity,BIGINT,YES,None,None,None


In [11]:
qry = """ 
    SELECT COUNT(*) AS total_rows,
    SUM(CASE WHEN returndate IS NULL THEN 1 ELSE 0 END) AS returndate_missing,
    SUM(CASE WHEN territorykey IS NULL THEN 1 ELSE 0 END) AS territorykey_missing,
    SUM(CASE WHEN productkey IS NULL THEN 1 ELSE 0 END) AS productkey_missing,
    SUM(CASE WHEN returnquantity IS NULL THEN 1 ELSE 0 END) AS returnquantity_missing
    FROM returns_data;
"""

conn.execute(qry).fetch_df()

,total_rows,returndate_missing,territorykey_missing,productkey_missing,returnquantity_missing
0,1809,0.0,0.0,0.0,0.0


### 7. Sales 2020 Table

In [12]:
conn.execute("DESCRIBE sales_2020").fetch_df()

,column_name,column_type,null,key,default,extra
0,orderdate,DATE,YES,None,None,None
1,stockdate,DATE,YES,None,None,None
2,ordernumber,VARCHAR,YES,None,None,None
3,productkey,BIGINT,YES,None,None,None
4,customerkey,BIGINT,YES,None,None,None
5,territorykey,BIGINT,YES,None,None,None
6,orderlineitem,BIGINT,YES,None,None,None
7,orderquantity,BIGINT,YES,None,None,None


In [13]:
qry = """ 
    SELECT COUNT(*) AS total_rows,
    SUM(CASE WHEN orderdate IS NULL THEN 1 ELSE 0 END) AS orderdate_missing,
    SUM(CASE WHEN stockdate IS NULL THEN 1 ELSE 0 END) AS stockdate_missing,
    SUM(CASE WHEN ordernumber IS NULL THEN 1 ELSE 0 END) AS ordernumber_missing,
    SUM(CASE WHEN productkey IS NULL THEN 1 ELSE 0 END) AS productkey_missing,
    SUM(CASE WHEN customerkey IS NULL THEN 1 ELSE 0 END) AS customerkey_missing,
    SUM(CASE WHEN territorykey IS NULL THEN 1 ELSE 0 END) AS territorykey_missing,
    SUM(CASE WHEN orderlineitem IS NULL THEN 1 ELSE 0 END) AS orderlineitem_missing,
    SUM(CASE WHEN orderquantity IS NULL THEN 1 ELSE 0 END) AS orderquantity_missing
    FROM sales_2020;
"""

conn.execute(qry).fetch_df()

,total_rows,orderdate_missing,stockdate_missing,ordernumber_missing,productkey_missing,customerkey_missing,territorykey_missing,orderlineitem_missing,orderquantity_missing
0,2630,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 8. Sales 2021 Table

In [14]:
conn.execute("DESCRIBE sales_2021").fetch_df()

,column_name,column_type,null,key,default,extra
0,orderdate,DATE,YES,None,None,None
1,stockdate,DATE,YES,None,None,None
2,ordernumber,VARCHAR,YES,None,None,None
3,productkey,BIGINT,YES,None,None,None
4,customerkey,BIGINT,YES,None,None,None
5,territorykey,BIGINT,YES,None,None,None
6,orderlineitem,BIGINT,YES,None,None,None
7,orderquantity,BIGINT,YES,None,None,None


In [15]:
qry = """ 
    SELECT COUNT(*) AS total_rows,
    SUM(CASE WHEN orderdate IS NULL THEN 1 ELSE 0 END) AS orderdate_missing,
    SUM(CASE WHEN stockdate IS NULL THEN 1 ELSE 0 END) AS stockdate_missing,
    SUM(CASE WHEN ordernumber IS NULL THEN 1 ELSE 0 END) AS ordernumber_missing,
    SUM(CASE WHEN productkey IS NULL THEN 1 ELSE 0 END) AS productkey_missing,
    SUM(CASE WHEN customerkey IS NULL THEN 1 ELSE 0 END) AS customerkey_missing,
    SUM(CASE WHEN territorykey IS NULL THEN 1 ELSE 0 END) AS territorykey_missing,
    SUM(CASE WHEN orderlineitem IS NULL THEN 1 ELSE 0 END) AS orderlineitem_missing,
    SUM(CASE WHEN orderquantity IS NULL THEN 1 ELSE 0 END) AS orderquantity_missing
    FROM sales_2021;
"""

conn.execute(qry).fetch_df()

,total_rows,orderdate_missing,stockdate_missing,ordernumber_missing,productkey_missing,customerkey_missing,territorykey_missing,orderlineitem_missing,orderquantity_missing
0,23935,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 9. Sales 2022 Table

In [16]:
conn.execute("DESCRIBE sales_2022").fetch_df()

,column_name,column_type,null,key,default,extra
0,orderdate,DATE,YES,None,None,None
1,stockdate,DATE,YES,None,None,None
2,ordernumber,VARCHAR,YES,None,None,None
3,productkey,BIGINT,YES,None,None,None
4,customerkey,BIGINT,YES,None,None,None
5,territorykey,BIGINT,YES,None,None,None
6,orderlineitem,BIGINT,YES,None,None,None
7,orderquantity,BIGINT,YES,None,None,None


In [17]:
qry = """ 
    SELECT COUNT(*) AS total_rows,
    SUM(CASE WHEN orderdate IS NULL THEN 1 ELSE 0 END) AS orderdate_missing,
    SUM(CASE WHEN stockdate IS NULL THEN 1 ELSE 0 END) AS stockdate_missing,
    SUM(CASE WHEN ordernumber IS NULL THEN 1 ELSE 0 END) AS ordernumber_missing,
    SUM(CASE WHEN productkey IS NULL THEN 1 ELSE 0 END) AS productkey_missing,
    SUM(CASE WHEN customerkey IS NULL THEN 1 ELSE 0 END) AS customerkey_missing,
    SUM(CASE WHEN territorykey IS NULL THEN 1 ELSE 0 END) AS territorykey_missing,
    SUM(CASE WHEN orderlineitem IS NULL THEN 1 ELSE 0 END) AS orderlineitem_missing,
    SUM(CASE WHEN orderquantity IS NULL THEN 1 ELSE 0 END) AS orderquantity_missing
    FROM sales_2022;
"""

conn.execute(qry).fetch_df()

,total_rows,orderdate_missing,stockdate_missing,ordernumber_missing,productkey_missing,customerkey_missing,territorykey_missing,orderlineitem_missing,orderquantity_missing
0,29481,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 10. Territory Lookup Table

In [18]:
conn.execute("DESCRIBE territory_lookup").fetch_df()

,column_name,column_type,null,key,default,extra
0,salesterritorykey,BIGINT,YES,None,None,None
1,region,VARCHAR,YES,None,None,None
2,country,VARCHAR,YES,None,None,None
3,continent,VARCHAR,YES,None,None,None


In [19]:
qry = """ 
    SELECT COUNT(*) AS total_rows,
    SUM(CASE WHEN salesterritorykey IS NULL THEN 1 ELSE 0 END) AS salesterritorykey_missing,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS region_missing,
    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS country_missing,
    SUM(CASE WHEN continent IS NULL THEN 1 ELSE 0 END) AS continent_missing
    FROM territory_lookup;
"""

conn.execute(qry).fetch_df()

,total_rows,salesterritorykey_missing,region_missing,country_missing,continent_missing
0,10,0.0,0.0,0.0,0.0


## 3. Duplicates

### 1. Calendar Lookup Dup

In [2]:
qry = """ 
    SELECT date, COUNT(*) AS dup_count
    FROM calendar_lookup
    GROUP BY date
    HAVING COUNT(*) > 1;
"""

conn.execute(qry).fetch_df()

,date,dup_count


### 2. Customer Lookup Dup

In [4]:
qry = """ 
    SELECT customerkey, emailaddress, COUNT(*) AS dup_count,
    FROM customer_lookup
    GROUP BY customerkey, emailaddress
    HAVING COUNT(*) > 1;
"""

conn.execute(qry).fetch_df()

,customerkey,emailaddress,dup_count
0,30---,None,3


In [5]:
qry = """ 
    DELETE FROM customer_lookup
    WHERE ROWID NOT IN (
        SELECT MIN(ROWID)
        FROM customer_lookup
        GROUP BY customerkey, emailaddress
    );
"""

conn.execute(qry).fetch_df()

,Count
0,2


### 3. Product Categories Lookup Dup

In [8]:
qry = """ 
    SELECT productcategorykey, COUNT(*) AS dup_count,
    FROM product_categories_lookup
    GROUP BY productcategorykey
    HAVING COUNT(*) > 1;
"""

conn.execute(qry).fetch_df()

,productcategorykey,dup_count


### 4. Product Lookup Dup

In [10]:
qry = """ 
    SELECT productkey, productsku, COUNT(*) AS dup_count,
    FROM product_lookup
    GROUP BY productkey, productsku
    HAVING COUNT(*) > 1;
"""

conn.execute(qry).fetch_df()

,productkey,productsku,dup_count


### 5. Product SubCategories Lookup Dup

In [12]:
qry = """ 
    SELECT productsubcategorykey, COUNT(*) AS dup_count
    FROM product_subcategories_lookup
    GROUP BY productsubcategorykey
    HAVING COUNT(*) > 1;
"""

conn.execute(qry).fetch_df()

,productsubcategorykey,dup_count


### 6. Returns Data Dup

In [14]:
qry = """ 
    SELECT returndate, territorykey, productkey, COUNT(*) AS dup_count
    FROM returns_data
    GROUP BY returndate, territorykey, productkey
    HAVING COUNT(*) > 1;
"""

conn.execute(qry).fetch_df()

,returndate,territorykey,productkey,dup_count


### 7. Sales Data 2020, 2021, 2022

<h3>2020</h3>

In [18]:
qry = """ 
    SELECT ordernumber, orderlineitem, productkey, customerkey, COUNT(*) AS dup_count
    FROM sales_2020
    GROUP BY ordernumber, orderlineitem, productkey, customerkey
    HAVING COUNT(*) > 1
    ORDER BY dup_count DESC;
"""

conn.execute(qry).fetch_df()

,ordernumber,orderlineitem,productkey,customerkey,dup_count


<h3>2021</h3>

In [19]:
qry = """ 
    SELECT ordernumber, orderlineitem, productkey, customerkey, COUNT(*) AS dup_count
    FROM sales_2021
    GROUP BY ordernumber, orderlineitem, productkey, customerkey
    HAVING COUNT(*) > 1
    ORDER BY dup_count DESC;
"""

conn.execute(qry).fetch_df()

,ordernumber,orderlineitem,productkey,customerkey,dup_count


<h3>2022</h3>

In [20]:
qry = """ 
    SELECT ordernumber, orderlineitem, productkey, customerkey, COUNT(*) AS dup_count
    FROM sales_2022
    GROUP BY ordernumber, orderlineitem, productkey, customerkey
    HAVING COUNT(*) > 1
    ORDER BY dup_count DESC;
"""

conn.execute(qry).fetch_df()

,ordernumber,orderlineitem,productkey,customerkey,dup_count


### 8. Territory Lookup Dup

In [21]:
conn.execute("DESCRIBE territory_lookup").fetch_df()

,column_name,column_type,null,key,default,extra
0,salesterritorykey,BIGINT,YES,None,None,None
1,region,VARCHAR,YES,None,None,None
2,country,VARCHAR,YES,None,None,None
3,continent,VARCHAR,YES,None,None,None


In [22]:
qry = """ 
    SELECT salesterritorykey, COUNT(*) dup_count
    FROM territory_lookup
    GROUP BY salesterritorykey
    HAVING COUNT(*) > 1;
"""

conn.execute(qry).fetch_df()

,salesterritorykey,dup_count


# 2. EDA